# Smart Traffic Prediction & Optimization System
**LSTM + Transformer | Congestion Forecasting | Signal Optimization | Gradio Dashboard**

In [ ]:
!pip install gradio folium -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import folium
from folium.plugins import HeatMap
import gradio as gr
import json, math, random
from datetime import datetime

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Load & Explore Data

In [ ]:
PATH = '/kaggle/input/datasets/pooriamst/metro-interstate-traffic-volume/Metro_Interstate_Traffic_Volume.csv'

df = pd.read_csv(PATH)
df['date_time'] = pd.to_datetime(df['date_time'])
df = df.sort_values('date_time').reset_index(drop=True)
df = df.drop_duplicates(subset='date_time').reset_index(drop=True)

print(df.shape)
df.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.patch.set_facecolor('#0f0f1a')
for ax in axes.flat:
    ax.set_facecolor('#1a1a2e')
    ax.tick_params(colors='#aaaacc')
    ax.spines[:].set_color('#333366')

sample = df[df['date_time'].dt.year == 2017].copy()
axes[0,0].plot(sample['date_time'], sample['traffic_volume'], color='#00d4ff', lw=0.6, alpha=0.8)
axes[0,0].set_title('Traffic Volume — 2017', color='white', fontsize=13)
axes[0,0].set_xlabel('Date', color='#aaaacc')
axes[0,0].set_ylabel('Volume', color='#aaaacc')

hourly = df.groupby(df['date_time'].dt.hour)['traffic_volume'].mean()
axes[0,1].bar(hourly.index, hourly.values, color='#7b2fff', edgecolor='#9d5fff', linewidth=0.5)
axes[0,1].set_title('Avg Traffic by Hour', color='white', fontsize=13)
axes[0,1].set_xlabel('Hour', color='#aaaacc')
axes[0,1].set_ylabel('Volume', color='#aaaacc')

dow = df.groupby(df['date_time'].dt.dayofweek)['traffic_volume'].mean()
days = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
colors = ['#ff6b6b' if i < 5 else '#ffd93d' for i in range(7)]
axes[1,0].bar(days, dow.values, color=colors, edgecolor='#ffffff22', linewidth=0.5)
axes[1,0].set_title('Avg Traffic by Day of Week', color='white', fontsize=13)
axes[1,0].set_xlabel('Day', color='#aaaacc')
axes[1,0].set_ylabel('Volume', color='#aaaacc')

axes[1,1].hist(df['traffic_volume'], bins=60, color='#00ffaa', edgecolor='#003322', linewidth=0.4, alpha=0.85)
axes[1,1].set_title('Traffic Volume Distribution', color='white', fontsize=13)
axes[1,1].set_xlabel('Volume', color='#aaaacc')
axes[1,1].set_ylabel('Count', color='#aaaacc')

plt.suptitle('Exploratory Data Analysis', color='white', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

## 2. Feature Engineering

In [ ]:
def engineer_features(df):
    d = df.copy()
    dt = d['date_time']

    d['hour']        = dt.dt.hour
    d['dayofweek']   = dt.dt.dayofweek
    d['month']       = dt.dt.month
    d['is_weekend']  = (d['dayofweek'] >= 5).astype(int)
    d['is_rush']     = d['hour'].apply(lambda h: 1 if (7<=h<=9 or 16<=h<=19) else 0)
    d['is_night']    = d['hour'].apply(lambda h: 1 if (h<6 or h>=22) else 0)

    d['hour_sin']    = np.sin(2*np.pi*d['hour']/24)
    d['hour_cos']    = np.cos(2*np.pi*d['hour']/24)
    d['dow_sin']     = np.sin(2*np.pi*d['dayofweek']/7)
    d['dow_cos']     = np.cos(2*np.pi*d['dayofweek']/7)
    d['month_sin']   = np.sin(2*np.pi*d['month']/12)
    d['month_cos']   = np.cos(2*np.pi*d['month']/12)

    d['weather_main'] = d['weather_main'].astype('category').cat.codes
    d['holiday']      = (d['holiday'] != 'None').astype(int)

    for lag in [1, 2, 3, 6, 12, 24]:
        d[f'lag_{lag}'] = d['traffic_volume'].shift(lag)

    d['roll_mean_6']  = d['traffic_volume'].shift(1).rolling(6).mean()
    d['roll_std_6']   = d['traffic_volume'].shift(1).rolling(6).std()
    d['roll_mean_24'] = d['traffic_volume'].shift(1).rolling(24).mean()

    d = d.dropna().reset_index(drop=True)
    return d

df_feat = engineer_features(df)
print(f'Features: {df_feat.shape[1]} | Rows: {df_feat.shape[0]}')
df_feat.head(3)

## 3. Dataset & Scaling

In [ ]:
FEATURE_COLS = [
    'hour_sin','hour_cos','dow_sin','dow_cos','month_sin','month_cos',
    'is_weekend','is_rush','is_night','holiday','weather_main',
    'temp','rain_1h','snow_1h','clouds_all',
    'lag_1','lag_2','lag_3','lag_6','lag_12','lag_24',
    'roll_mean_6','roll_std_6','roll_mean_24'
]
TARGET = 'traffic_volume'

SEQ_LEN   = 24
PRED_LEN  = 1
TRAIN_PCT = 0.8

X_raw = df_feat[FEATURE_COLS].values
y_raw = df_feat[TARGET].values.reshape(-1,1)

split = int(len(X_raw) * TRAIN_PCT)

x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_train_s = x_scaler.fit_transform(X_raw[:split])
X_test_s  = x_scaler.transform(X_raw[split:])
y_train_s = y_scaler.fit_transform(y_raw[:split])
y_test_s  = y_scaler.transform(y_raw[split:])

class TrafficDataset(Dataset):
    def __init__(self, X, y, seq_len):
        self.X, self.y, self.seq_len = X, y, seq_len
    def __len__(self):
        return len(self.X) - self.seq_len
    def __getitem__(self, i):
        x = torch.FloatTensor(self.X[i:i+self.seq_len])
        t = torch.FloatTensor(self.y[i+self.seq_len])
        return x, t

BATCH = 64
train_ds = TrafficDataset(X_train_s, y_train_s, SEQ_LEN)
test_ds  = TrafficDataset(X_test_s,  y_test_s,  SEQ_LEN)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_dl)} | Test batches: {len(test_dl)}')

## 4. Models — LSTM & Transformer

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden=128, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, layers, batch_first=True,
                            dropout=dropout, bidirectional=True)
        self.attn = nn.Linear(hidden*2, 1)
        self.head = nn.Sequential(
            nn.Linear(hidden*2, 64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        w = torch.softmax(self.attn(out), dim=1)
        ctx = (w * out).sum(dim=1)
        return self.head(ctx)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class TransformerModel(nn.Module):
    def __init__(self, input_dim, d_model=64, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.proj   = nn.Linear(input_dim, d_model)
        self.pe     = PositionalEncoding(d_model, dropout=dropout)
        enc_layer   = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=128,
                                                  dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers)
        self.head   = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = self.pe(self.proj(x))
        x = self.encoder(x)
        return self.head(x[:, -1])


INPUT_DIM = len(FEATURE_COLS)
lstm_model = LSTMModel(INPUT_DIM).to(DEVICE)
trf_model  = TransformerModel(INPUT_DIM).to(DEVICE)

total_lstm = sum(p.numel() for p in lstm_model.parameters())
total_trf  = sum(p.numel() for p in trf_model.parameters())
print(f'LSTM params: {total_lstm:,} | Transformer params: {total_trf:,}')

## 5. Training

In [ ]:
def train_model(model, train_dl, test_dl, epochs=30, lr=1e-3, name='Model'):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    criterion = nn.HuberLoss(delta=0.5)

    train_losses, val_losses = [], []
    best_val, patience, counter = float('inf'), 7, 0

    for epoch in range(epochs):
        model.train()
        tloss = 0
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tloss += loss.item()

        model.eval()
        vloss = 0
        with torch.no_grad():
            for xb, yb in test_dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                vloss += criterion(model(xb), yb).item()

        tl = tloss/len(train_dl)
        vl = vloss/len(test_dl)
        train_losses.append(tl)
        val_losses.append(vl)
        scheduler.step()

        if vl < best_val:
            best_val = vl
            counter = 0
            torch.save(model.state_dict(), f'{name}_best.pt')
        else:
            counter += 1
            if counter >= patience:
                print(f'Early stop at epoch {epoch+1}')
                break

        if (epoch+1) % 5 == 0:
            print(f'[{name}] Epoch {epoch+1:03d} | Train: {tl:.4f} | Val: {vl:.4f}')

    model.load_state_dict(torch.load(f'{name}_best.pt'))
    return train_losses, val_losses


print('Training LSTM...')
lstm_tl, lstm_vl = train_model(lstm_model, train_dl, test_dl, epochs=30, name='LSTM')

print('\nTraining Transformer...')
trf_tl, trf_vl = train_model(trf_model, train_dl, test_dl, epochs=30, name='TRF')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f1a')
for ax in axes:
    ax.set_facecolor('#1a1a2e')
    ax.tick_params(colors='#aaaacc')
    ax.spines[:].set_color('#333366')

for ax, tl, vl, title in zip(axes,
    [lstm_tl, trf_tl], [lstm_vl, trf_vl],
    ['LSTM Training', 'Transformer Training']):
    ax.plot(tl, label='Train', color='#00d4ff', lw=2)
    ax.plot(vl, label='Val',   color='#ff6b6b', lw=2)
    ax.set_title(title, color='white', fontsize=13)
    ax.set_xlabel('Epoch', color='#aaaacc')
    ax.set_ylabel('Loss',  color='#aaaacc')
    ax.legend(facecolor='#1a1a2e', labelcolor='white')

plt.suptitle('Loss Curves', color='white', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('loss_curves.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

## 6. Evaluation

In [ ]:
def evaluate(model, dl, y_scaler):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in dl:
            xb = xb.to(DEVICE)
            out = model(xb).cpu().numpy()
            preds.append(out)
            trues.append(yb.numpy())
    preds = y_scaler.inverse_transform(np.vstack(preds))
    trues = y_scaler.inverse_transform(np.vstack(trues))
    mae  = mean_absolute_error(trues, preds)
    rmse = np.sqrt(mean_squared_error(trues, preds))
    r2   = r2_score(trues, preds)
    mape = np.mean(np.abs((trues - preds) / (trues + 1e-8))) * 100
    return mae, rmse, r2, mape, preds.flatten(), trues.flatten()

lstm_mae, lstm_rmse, lstm_r2, lstm_mape, lstm_pred, lstm_true = evaluate(lstm_model, test_dl, y_scaler)
trf_mae,  trf_rmse,  trf_r2,  trf_mape,  trf_pred,  trf_true  = evaluate(trf_model,  test_dl, y_scaler)

results = pd.DataFrame({
    'Model' : ['LSTM (BiDir+Attn)', 'Transformer'],
    'MAE'   : [lstm_mae, trf_mae],
    'RMSE'  : [lstm_rmse, trf_rmse],
    'R²'    : [lstm_r2, trf_r2],
    'MAPE%' : [lstm_mape, trf_mape]
})
print(results.to_string(index=False))

In [ ]:
N = 500
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.patch.set_facecolor('#0f0f1a')
for ax in axes:
    ax.set_facecolor('#1a1a2e')
    ax.tick_params(colors='#aaaacc')
    ax.spines[:].set_color('#333366')

for ax, pred, name, col in zip(axes,
    [lstm_pred[:N], trf_pred[:N]],
    ['LSTM', 'Transformer'],
    ['#00d4ff', '#7b2fff']):
    ax.plot(lstm_true[:N], label='Actual',    color='#ff6b6b', lw=1.2, alpha=0.8)
    ax.plot(pred,          label=f'{name} Pred', color=col,   lw=1.0, alpha=0.85)
    ax.set_title(f'{name} — Predictions vs Actual', color='white', fontsize=13)
    ax.set_ylabel('Traffic Volume', color='#aaaacc')
    ax.legend(facecolor='#1a1a2e', labelcolor='white')

plt.tight_layout()
plt.savefig('predictions.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

## 7. Congestion Classification & Signal Optimization

In [ ]:
def classify_congestion(volume):
    if volume < 1500:  return 'Free Flow',   '#00ff88'
    if volume < 3000:  return 'Light',        '#aaff00'
    if volume < 4500:  return 'Moderate',     '#ffdd00'
    if volume < 5500:  return 'Heavy',        '#ff8800'
    return               'Severe',            '#ff2222'

def optimize_signal_timing(predicted_volume):
    level, _ = classify_congestion(predicted_volume)
    plans = {
        'Free Flow' : {'green': 45, 'yellow': 5, 'red': 25, 'cycle': 75},
        'Light'     : {'green': 50, 'yellow': 5, 'red': 25, 'cycle': 80},
        'Moderate'  : {'green': 55, 'yellow': 5, 'red': 20, 'cycle': 80},
        'Heavy'     : {'green': 65, 'yellow': 5, 'red': 15, 'cycle': 85},
        'Severe'    : {'green': 70, 'yellow': 5, 'red': 10, 'cycle': 85},
    }
    plan = plans[level]
    return {**plan, 'level': level, 'action': _action(level)}

def _action(level):
    return {
        'Free Flow' : 'Normal operations. No intervention.',
        'Light'     : 'Slight green extension on main corridor.',
        'Moderate'  : 'Activate adaptive cycle. Extend main green.',
        'Heavy'     : 'Max green time. Suppress cross-street phases.',
        'Severe'    : 'Emergency mode. Coordinate upstream signals. Alert navigation apps.',
    }[level]

def suggest_routes(hour, is_weekend, volume):
    level, _ = classify_congestion(volume)
    routes = {
        'Primary I-94 W' : {'base_time': 22, 'distance': 18.5},
        'Alt Route A'    : {'base_time': 28, 'distance': 21.2},
        'Alt Route B'    : {'base_time': 31, 'distance': 19.8},
        'Local Streets'  : {'base_time': 35, 'distance': 16.1},
    }
    multipliers = {'Free Flow':1.0,'Light':1.2,'Moderate':1.5,'Heavy':1.9,'Severe':2.5}
    m = multipliers[level]
    rec = []
    for name, info in routes.items():
        delay = 0 if 'Alt' in name or 'Local' in name else (m - 1) * info['base_time']
        rec.append({'route': name, 'est_time': round(info['base_time'] + delay, 1),
                    'distance_mi': info['distance']})
    return sorted(rec, key=lambda x: x['est_time'])

vol_test = 4800
level, color = classify_congestion(vol_test)
plan  = optimize_signal_timing(vol_test)
routes = suggest_routes(hour=8, is_weekend=0, volume=vol_test)

print(f'Volume: {vol_test} → Congestion: {level}')
print(f'Signal: Green={plan["green"]}s  Yellow={plan["yellow"]}s  Red={plan["red"]}s')
print(f'Action: {plan["action"]}')
print('\nRoute Suggestions:')
for r in routes:
    print(f"  {r['route']}: {r['est_time']} min | {r['distance_mi']} mi")

## 8. Folium Map Visualization

In [ ]:
SEGMENTS = [
    {'name':'I-94 W Seg 1','lat':44.980,'lon':-93.270,'volume':5800},
    {'name':'I-94 W Seg 2','lat':44.978,'lon':-93.310,'volume':4900},
    {'name':'I-94 W Seg 3','lat':44.975,'lon':-93.350,'volume':3800},
    {'name':'I-94 W Seg 4','lat':44.972,'lon':-93.390,'volume':2800},
    {'name':'I-94 W Seg 5','lat':44.970,'lon':-93.430,'volume':1600},
    {'name':'Alt A Seg 1' ,'lat':44.990,'lon':-93.300,'volume':1200},
    {'name':'Alt A Seg 2' ,'lat':44.988,'lon':-93.340,'volume':1400},
    {'name':'Alt B Seg 1' ,'lat':44.965,'lon':-93.290,'volume':2100},
]

LEVEL_COLOR = {
    'Free Flow':'#00ff88','Light':'#aaff00',
    'Moderate':'#ffdd00','Heavy':'#ff8800','Severe':'#ff2222'
}

m = folium.Map(location=[44.975, -93.35], zoom_start=12,
               tiles='CartoDB dark_matter')

heat_data = []
for seg in SEGMENTS:
    level, _ = classify_congestion(seg['volume'])
    col = LEVEL_COLOR[level]
    plan = optimize_signal_timing(seg['volume'])
    routes = suggest_routes(8, 0, seg['volume'])

    popup_html = f"""
    <div style='font-family:monospace;background:#1a1a2e;color:white;padding:10px;border-radius:8px;min-width:220px'>
      <b style='color:{col}'>{seg['name']}</b><br>
      Volume: <b>{seg['volume']:,}</b><br>
      Status: <b style='color:{col}'>{level}</b><br>
      Signal: 🟢{plan['green']}s 🟡{plan['yellow']}s 🔴{plan['red']}s<br>
      Best Route: {routes[0]['route']} ({routes[0]['est_time']} min)<br>
      <i style='color:#aaa;font-size:11px'>{plan['action']}</i>
    </div>
    """
    folium.CircleMarker(
        location=[seg['lat'], seg['lon']],
        radius=max(8, seg['volume']//500),
        color=col, fill=True, fill_color=col, fill_opacity=0.7,
        popup=folium.Popup(popup_html, max_width=260)
    ).add_to(m)

    weight = seg['volume'] / 6000.0
    heat_data.append([seg['lat'], seg['lon'], weight])

HeatMap(heat_data, radius=40, blur=25,
        gradient={'0.2':'blue','0.5':'lime','0.8':'orange','1.0':'red'}).add_to(m)

legend_html = """
<div style='position:fixed;bottom:30px;left:30px;z-index:1000;
     background:#0f0f1a;padding:12px 16px;border-radius:10px;
     border:1px solid #333;font-family:monospace;color:white;'>
  <b>Congestion Level</b><br>
  <span style='color:#00ff88'>●</span> Free Flow (&lt;1500)<br>
  <span style='color:#aaff00'>●</span> Light (1500-3000)<br>
  <span style='color:#ffdd00'>●</span> Moderate (3000-4500)<br>
  <span style='color:#ff8800'>●</span> Heavy (4500-5500)<br>
  <span style='color:#ff2222'>●</span> Severe (&gt;5500)
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m.save('traffic_map.html')
print('Map saved → traffic_map.html')
m

## 9. Gradio Dashboard

In [ ]:
best_model = lstm_model if lstm_r2 >= trf_r2 else trf_model
best_model.eval()

RECENT_X = X_test_s[-SEQ_LEN:]

def predict_traffic(hour, day_of_week, month, is_holiday,
                    weather, temp_f, rain, clouds):
    hour       = int(hour)
    dow        = int(day_of_week)
    month      = int(month)
    temp_k     = (float(temp_f) - 32) / 1.8 + 273.15
    rain_mm    = float(rain)
    clouds_pct = float(clouds)
    weather_c  = int(weather)
    holiday_i  = 1 if is_holiday else 0

    seq = torch.FloatTensor(RECENT_X).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        raw = best_model(seq).cpu().numpy()
    volume = float(y_scaler.inverse_transform(raw)[0][0])

    is_rush    = 1 if (7<=hour<=9 or 16<=hour<=19) else 0
    is_weekend = 1 if dow >= 5 else 0
    volume    *= (1.15 if is_rush and not is_weekend else 1.0)
    volume    *= (0.80 if is_weekend else 1.0)
    volume    *= (0.85 if rain_mm > 5 else 1.0)
    volume     = max(100, min(7500, volume))

    level, color = classify_congestion(volume)
    plan   = optimize_signal_timing(volume)
    routes = suggest_routes(hour, is_weekend, volume)

    result = f"""## 🚦 Traffic Prediction Results

| Metric | Value |
|--------|-------|
| **Predicted Volume** | {volume:,.0f} vehicles/hr |
| **Congestion Level** | {level} |
| **Rush Hour** | {'Yes' if is_rush else 'No'} |
| **Weekend** | {'Yes' if is_weekend else 'No'} |

---
## 🔴🟡🟢 Signal Timing Optimization

| Phase | Duration |
|-------|----------|
| 🟢 Green | {plan['green']}s |
| 🟡 Yellow | {plan['yellow']}s |
| 🔴 Red | {plan['red']}s |
| Full Cycle | {plan['cycle']}s |

**Action:** {plan['action']}

---
## 🗺️ Route Suggestions

| Rank | Route | Est. Time | Distance |
|------|-------|-----------|----------|
"""
    medals = ['🥇','🥈','🥉','  4.']
    for i, r in enumerate(routes[:4]):
        result += f"| {medals[i]} | {r['route']} | {r['est_time']} min | {r['distance_mi']} mi |\n"

    return result


WEATHER_CODES = ['Clear(0)','Clouds(1)','Drizzle(2)','Rain(3)','Thunderstorm(4)','Snow(5)','Mist(6)']

with gr.Blocks(theme=gr.themes.Base(), title='Smart Traffic System') as demo:
    gr.Markdown("""# 🚦 Smart Traffic Prediction & Optimization System
    **LSTM + Transformer | Congestion Forecasting | Signal Optimization**""")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### ⏰ Time & Context')
            hour    = gr.Slider(0, 23, value=8,  step=1, label='Hour of Day')
            dow     = gr.Slider(0, 6,  value=1,  step=1, label='Day of Week (0=Mon, 6=Sun)')
            month   = gr.Slider(1, 12, value=6,  step=1, label='Month')
            holiday = gr.Checkbox(label='Is Holiday?', value=False)

            gr.Markdown('### 🌤️ Weather')
            weather = gr.Dropdown(choices=list(range(7)), value=0, label='Weather Code',
                                  info='0=Clear 1=Clouds 2=Drizzle 3=Rain 4=Thunder 5=Snow 6=Mist')
            temp    = gr.Slider(0, 110, value=65, step=1, label='Temperature (°F)')
            rain    = gr.Slider(0, 50,  value=0,  step=0.5, label='Rain (mm/hr)')
            clouds  = gr.Slider(0, 100, value=20, step=5, label='Cloud Cover (%)')

            btn = gr.Button('🔮 Predict & Optimize', variant='primary')

        with gr.Column(scale=2):
            out = gr.Markdown()

    btn.click(predict_traffic,
              inputs=[hour, dow, month, holiday, weather, temp, rain, clouds],
              outputs=out)

    gr.Examples(
        examples=[[8,0,9,False,0,68,0,20],[17,4,12,False,1,55,0,60],
                  [3,6,1,False,0,20,0,10],[8,0,7,True,3,50,15,90]],
        inputs=[hour, dow, month, holiday, weather, temp, rain, clouds],
        label='Quick Examples (Rush Hour | Evening | Night | Holiday Rain)'
    )

demo.launch(share=True)

## 10. Save Artifacts

In [ ]:
torch.save(lstm_model.state_dict(), 'lstm_traffic.pt')
torch.save(trf_model.state_dict(),  'transformer_traffic.pt')

import pickle
with open('scalers.pkl', 'wb') as f:
    pickle.dump({'x': x_scaler, 'y': y_scaler}, f)

results.to_csv('model_results.csv', index=False)

print('All artifacts saved.')
print(results.to_string(index=False))